# LP-ResMIL — fully nested participant-level evaluation

Only LP-ResMIL is trained. This notebook retains its multiscale separable-convolution window encoder, channel gating, and masked gated MIL pooling over recording windows. It does not retrain LPGNet or other reference models.

## Keras 3 masking fix
The masked `tf.where`, remasking and weighted-sum Lambdas are replaced with registered serializable layers with explicit output shapes. This is gated attention pooling, not Q/K/V self-attention. Valid windows are gathered **before** the encoder, so padding cannot influence trainable BatchNorm statistics. Masked weights sum to one over valid windows; an all-padded recording is rejected.

## Evaluation and checkpoints
- Ten outer participant folds; three inner folds for configuration assessment; a separate participant early-stopping core in each inner training partition.
- LP fitting uses core controls only during selection. Fixed-epoch final refitting may use all outer-training controls, never outer test controls.
- Curated checksum-verified original data and exact-ID clinical joins. Metadata are used only after prediction.
- Participant-balanced window/recording objectives; participant OOF predictions average recording probabilities.
- Real forward/backward, padding and safe model/optimizer serialization smoke tests.
- Atomic checkpoints after every completed epoch and inner/outer task; immutable run plan, no CSV-based guessing of completion.

`RUN_EXPERIMENT=False` initially. Choose `CHECKPOINT_DIR` before training; `MAX_NEW_TASKS=1` limits work per invocation. Cloud working storage is not durable across a full runtime reset: export the checkpoint bundle or use mounted persistent storage. Old checkpoints from the Lambda/padding implementation are not reused.

In [ ]:
# Run this setup first in a fresh Kaggle/Colab kernel.
import os, sys, tempfile, subprocess
from pathlib import Path
from collections import Counter
os.environ.setdefault('TF_FORCE_GPU_ALLOW_GROWTH','true')
SOURCE_REV = '5c8bd0df73870c03baf9d0f7252a7eeb73131328'
REPO_URL = 'https://github.com/AlexM3020/Subject-Independent-Evaluation-of-Mamba-for-Parkinson-s-Disease-Detection-from-Gait-Signals.git'
INSTALL_DEPENDENCIES = False
roots=[Path.cwd(),*Path.cwd().parents]
REPO_ROOT=next((p for b in roots for p in (b,b/'paper_repository') if (p/'gait/tf_notebook_support.py').is_file()),None)
if REPO_ROOT is None:
    REPO_ROOT=Path(tempfile.gettempdir())/('gait-keras-'+SOURCE_REV[:12])
    if not REPO_ROOT.exists(): subprocess.run(['git','clone','--no-checkout',REPO_URL,str(REPO_ROOT)],check=True)
    subprocess.run(['git','-C',str(REPO_ROOT),'checkout','--detach',SOURCE_REV],check=True)
REPO_ROOT=REPO_ROOT.resolve()
for filename in ('__init__.py','official_data.py','tf_notebook_support.py'):
    expected=subprocess.run(['git','-C',str(REPO_ROOT),'show',SOURCE_REV+':gait/'+filename],capture_output=True,check=True).stdout
    assert (REPO_ROOT/'gait'/filename).read_bytes().replace(b'\r\n',b'\n')==expected.replace(b'\r\n',b'\n'), 'Pinned source changed: '+filename
stale=[name for name,module in list(sys.modules.items()) if (name=='gait' or name.startswith('gait.')) and getattr(module,'__file__',None) and not Path(module.__file__).resolve().is_relative_to(REPO_ROOT)]
if stale: raise RuntimeError('Older gait imports cached: export needed results, restart kernel and run this setup first. '+str(stale))
sys.path.insert(0,str(REPO_ROOT))
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable,'-m','pip','install','-r',str(REPO_ROOT/'requirements-tensorflow.txt')],check=True)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
keras=tf.keras
from scipy.signal import lfilter
from sklearn.metrics import (roc_auc_score,accuracy_score,balanced_accuracy_score,f1_score,confusion_matrix,ConfusionMatrixDisplay,log_loss,roc_curve)
from sklearn.calibration import calibration_curve
from IPython.display import display
from gait import tf_notebook_support as nb
FAMILY='LP_ResMIL'
SEED,FOLD_SEED,SPLIT_SEED,SEARCH_SEED=1,0,9137,20261117
N_FOLDS,N_INNER,VAL_FRAC=10,3,.15
LP_ORDER,WIN,STRIDE=11,100,50
BATCH_WIN,BATCH_REC,PRED_BATCH=128,8,8
LR_WIN,LABEL_SMOOTH=5e-4,.1
MAX_EPOCHS_WIN,MAX_EPOCHS_MIL=1000,200
PATIENCE,RLR_PATIENCE,MIN_LR=25,14,1e-7
THRESHOLD=.5
WORK_ROOT=Path('/kaggle/working') if Path('/kaggle').exists() else Path('/content') if Path('/content').exists() else Path.cwd()
# For Colab persistence mount Drive yourself, then choose a Drive path here.
CHECKPOINT_DIR=WORK_ROOT/'lp_resmil_keras3_v2'
OUTDIR=CHECKPOINT_DIR/'reports'
CACHE_DIR=Path(tempfile.gettempdir())/'gaitpdb-1.0.0-verified-cache'
COHORT_POLICY='curated'
ALLOW_DOWNLOAD=True
RUN_EXPERIMENT=False
MAX_NEW_TASKS=1
VERBOSE_EPOCHS=1
RESTORE_FROM_ZIP=''
EXPERIMENT_COMPLETE=False
summary=participant_ci=participant_fold_metrics=selection_table=trial_table=nested_oof=pd.DataFrame()
participants=participant_errors=pd_severity=by_study=seed_stability=inner_loss=attention_audit=attention_oof=pd.DataFrame()
keras.utils.set_random_seed(SEED)
print('Pinned source:',SOURCE_REV,'| TensorFlow:',tf.__version__,'| Keras:',keras.__version__)
print('GPU:',tf.config.list_physical_devices('GPU'))
print('Checkpoint directory:',CHECKPOINT_DIR)
print('Working storage is NOT guaranteed to survive a full Kaggle/Colab runtime reset. Export a checkpoint bundle or use persistent storage.')

In [ ]:
TRAIN_SEEDS = [42, 123, 2026]
N_TRIALS = 12  # Unique candidates; shared across training seeds, selected inside inner CV.
N_BOOT = 2000
EXPORT_ATTENTION = True
REFERENCE_OOF_PATH = Path('lpgnet_variant_a_keras3_v2/reports/recording_oof.csv')
print('LP-ResMIL only; no reference model is retrained.')
print('Outer tasks:',len(TRAIN_SEEDS)*N_FOLDS,'| inner two-stage fits:',len(TRAIN_SEEDS)*N_FOLDS*N_TRIALS*N_INNER)
print('Additional outer-development epoch selection and fixed-epoch refit are performed per task.')

In [ ]:
records,clinical,data_audit,data_provenance = nb.load_inputs(CACHE_DIR,COHORT_POLICY,ALLOW_DOWNLOAD)
subjects=sorted({r['subject'] for r in records})
subj_label={r['subject']:r['label'] for r in records}
by_file={r['file']:i for i,r in enumerate(records)}
by_subject={s:[i for i,r in enumerate(records) if r['subject']==s] for s in subjects}
print('Cohort policy:',COHORT_POLICY)
print(len(records),'recordings |',len(subjects),'participants |',dict(Counter(subj_label.values())))
print('Data provenance:',data_provenance)
print('Clinical unknowns are retained; no severity is inferred from filenames.')

In [ ]:
def preprocess(x):
    return lfilter([.5,.5],1,x,axis=0)[::2]

def fit_lp(control_sigs,order=LP_ORDER):
    return nb.fit_lp(control_sigs,order)

def apply_lp(x,C,stds):
    return nb.apply_lp(x,C,stds)

def lp_features(indices,C,stds):
    return {i:apply_lp(records[i]['proc'],C,stds) for i in indices}

for r in records:
    r['proc']=preprocess(r['signal'])
print('Only downsampling is prepared globally. LP coefficients/scales are fitted on each early-stopping core, then refitted on outer training only after epoch selection.')

In [ ]:
def make_folds():
    return nb.outer_folds(records,N_FOLDS,FOLD_SEED)

def split_val(indices,frac=VAL_FRAC,seed=SEED):
    return nb.subject_split(records,indices,frac,seed)

def labels(indices):
    return np.array([records[i]['label'] for i in indices],np.float32)

FOLDS=make_folds()
for k,f in enumerate(FOLDS):
    print('Fold',k,'train participants:',len({records[i]['subject'] for i in f['train']}),
          'test participants:',len({records[i]['subject'] for i in f['test']}))

In [ ]:
# ============================================================
# OOF metrics and participant-level diagnostics
# ============================================================
def _validated_binary_inputs(y, p):
    y = np.asarray(y)
    p = np.asarray(p)
    if y.ndim != 1 or p.ndim != 1 or len(y) == 0 or len(y) != len(p):
        raise ValueError('Labels and probabilities must be nonempty 1-D arrays of equal length.')
    if np.iscomplexobj(y) or not np.isin(y, [0, 1]).all():
        raise ValueError('Labels must contain only binary values 0 and 1.')
    if np.iscomplexobj(p):
        raise ValueError('Probabilities must be real numbers in [0, 1].')
    p = np.asarray(p, float)
    if not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError('Probabilities must be finite and in [0, 1].')
    return y.astype(int), p

def metric_dict(y,p):
    y,p=_validated_binary_inputs(y,p)
    yh=(p>=THRESHOLD).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,yh,labels=[0,1]).ravel()
    return dict(n=len(y),accuracy=accuracy_score(y,yh),
                balanced_accuracy=balanced_accuracy_score(y,yh),
                macro_f1=f1_score(y,yh,labels=[0,1],average='macro',zero_division=0),
                f1_pd=f1_score(y,yh,zero_division=0),
                sensitivity=tp/(tp+fn) if tp+fn else np.nan,
                specificity=tn/(tn+fp) if tn+fp else np.nan,
                roc_auc=roc_auc_score(y,p) if len(np.unique(y))==2 else np.nan,
                log_loss=log_loss(y,np.clip(p,1e-7,1-1e-7),labels=[0,1]),
                brier=float(np.mean((p-y)**2)))

def oof_frame(rows):
    df=pd.DataFrame(rows)
    y,p=_validated_binary_inputs(df['label'],df['probability'])
    df['label']=y; df['probability']=p
    df['prediction']=(df['probability']>=THRESHOLD).astype(int)
    df['correct']=(df['prediction']==df['label']).astype(int)
    df['confidence']=np.maximum(df['probability'],1-df['probability'])
    df['nll']=-(df['label']*np.log(np.clip(df['probability'],1e-7,1))+
                (1-df['label'])*np.log(np.clip(1-df['probability'],1e-7,1)))
    # Strict metadata lookup: missing subjects or fields raise KeyError.
    for col in ('study','diagnosis','hy','hy_label','hy_status','age','gender'):
        df[col]=df['subject'].map(lambda s:clinical[s][col])
    return df

def participant_oof(df,group_cols):
    """Average recording probabilities so each participant has equal weight."""
    keys=group_cols+['subject']
    df=df.copy()
    y,p=_validated_binary_inputs(df['label'],df['probability'])
    df['label']=y; df['probability']=p
    if df[keys+['fold']].isna().any().any():
        raise ValueError('Participant grouping keys and folds must not be missing.')
    grouped=df.groupby(keys,as_index=False)
    counts=grouped[['label','fold']].nunique(dropna=False)
    if (counts['label']>1).any():
        raise ValueError('Conflicting labels for a subject within an evaluation group.')
    if (counts['fold']>1).any():
        raise ValueError('Multiple folds for a subject within an evaluation group.')
    if 'file' in df.columns and df.duplicated(subset=group_cols+['file']).any():
        raise ValueError('Duplicate file within an evaluation group.')
    agg=dict(label=('label','first'),fold=('fold','first'),probability=('probability','mean'),
             recordings=('file','nunique') if 'file' in df.columns else ('subject','size'))
    out=grouped.agg(**agg)
    return oof_frame(out.to_dict('records'))

def summarize_oof(df,group_cols):
    rows=[]
    for keys,g in df.groupby(group_cols):
        if not isinstance(keys,tuple): keys=(keys,)
        part=participant_oof(g,group_cols)
        row={**dict(zip(group_cols,keys)),
             **{'recording_'+k:v for k,v in metric_dict(g['label'],g['probability']).items()}}
        row.update({'participant_'+k:v for k,v in metric_dict(part['label'],part['probability']).items()})
        rows.append(row)
    return pd.DataFrame(rows)

def wilson_interval(correct,n,z=1.96):
    if n==0: return (np.nan,np.nan)
    ph=correct/n; den=1+z*z/n
    center=(ph+z*z/(2*n))/den
    half=z*np.sqrt(ph*(1-ph)/n+z*z/(4*n*n))/den
    return center-half,center+half

def error_tables(df,group_cols):
    part=participant_oof(df,group_cols)
    errors=part[part['correct']==0].sort_values(group_cols+['nll'],
             ascending=[True]*len(group_cols)+[False])
    pd_rows=[]
    for keys,g in part[part['label']==1].groupby(group_cols+['hy_label'],dropna=False):
        if not isinstance(keys,tuple): keys=(keys,)
        n=len(g); correct=int(g['correct'].sum()); lo,hi=wilson_interval(correct,n)
        pd_rows.append({**dict(zip(group_cols+['hy_label'],keys)),
                        'pd_participants':n,'correct':correct,'errors':n-correct,
                        'pd_sensitivity':correct/n,'wilson_95_low':lo,'wilson_95_high':hi})
    study_rows=[]
    for keys,g in part.groupby(group_cols+['study','diagnosis']):
        if not isinstance(keys,tuple): keys=(keys,)
        study_rows.append({**dict(zip(group_cols+['study','diagnosis'],keys)),
                           **metric_dict(g['label'],g['probability'])})
    return part,errors,pd.DataFrame(pd_rows),pd.DataFrame(study_rows)

def show_table(df,title,limit=100):
    print('\n'+title); display(df.head(limit))

def pooled_bootstrap(df,group_cols,metrics=('accuracy','balanced_accuracy','macro_f1',
                     'sensitivity','specificity','roc_auc','log_loss','brier')):
    """Diagnosis-stratified participant bootstrap conditional on fitted OOF predictions."""
    rows=[]
    for keys,g in df.groupby(group_cols):
        if not isinstance(keys,tuple): keys=(keys,)
        part=participant_oof(g,group_cols).reset_index(drop=True)
        y=part['label'].to_numpy(); p=part['probability'].to_numpy()
        classes=[np.flatnonzero(y==c) for c in (0,1)]
        for metric in metrics:
            rng=np.random.default_rng(2026+sum(ord(ch) for ch in metric+str(keys))%10000)
            vals=[]
            for _ in range(N_BOOT):
                ii=np.concatenate([rng.choice(ix,len(ix),replace=True) for ix in classes])
                vals.append(metric_dict(y[ii],p[ii])[metric])
            rows.append({**dict(zip(group_cols,keys)),'metric':metric,
                         'estimate':metric_dict(y,p)[metric],
                         'bootstrap_95_low':np.nanpercentile(vals,2.5),
                         'bootstrap_95_high':np.nanpercentile(vals,97.5)})
    return pd.DataFrame(rows)

def plot_oof_diagnostics(df,group_cols):
    part=participant_oof(df,group_cols); groups=list(part.groupby(group_cols))
    fig,axes=plt.subplots(len(groups),3,figsize=(16,4.3*len(groups)),squeeze=False)
    for row,(keys,g) in enumerate(groups):
        if not isinstance(keys,tuple): keys=(keys,)
        title=' | '.join(f'{c}={v}' for c,v in zip(group_cols,keys))
        cm=confusion_matrix(g['label'],g['prediction'],labels=[0,1])
        ConfusionMatrixDisplay(cm,display_labels=['Control','PD']).plot(
            ax=axes[row,0],colorbar=False)
        axes[row,0].set_title(title+' | participant OOF')
        fpr,tpr,_=roc_curve(g['label'],g['probability'])
        axes[row,1].plot(fpr,tpr,label=f"AUC={roc_auc_score(g['label'],g['probability']):.3f}")
        axes[row,1].plot([0,1],[0,1],'--',color='gray')
        axes[row,1].set(xlabel='False-positive rate',ylabel='True-positive rate',title=title)
        axes[row,1].legend()
        frac,meanp=calibration_curve(g['label'],g['probability'],n_bins=8,strategy='quantile')
        axes[row,2].plot(meanp,frac,'o-',label='Observed')
        axes[row,2].plot([0,1],[0,1],'--',color='gray')
        axes[row,2].set(xlabel='Mean predicted probability',ylabel='Observed PD fraction',
                        xlim=(0,1),ylim=(0,1),title='Calibration (descriptive)')
        axes[row,2].legend()
    fig.tight_layout(); plt.show()

## Model and bounded search

The window encoder uses parallel separable convolutions, channel gating, dilated residual blocks and a temporal weighted sum. Recording-level gated attention pools only valid window embeddings. The frozen mode runs the encoder in inference mode; the `last` mode fine-tunes the declared last block and embedding layers.

The existing search ranges remain editable below. Each candidate is evaluated on all three inner assessment folds; early stopping occurs on separate core-held-out participants. Stage-1 and stage-2 epoch selection are checkpointed. The final outer model uses fixed training-only-selected epoch counts and never monitors the outer test set.

No attention weight is interpreted as a calibrated probability of clinical importance.

### Checkpoint safety

Only a committed epoch or task is resumed. Model/optimizer checkpoints and the saved run plan must remain together; do not edit completion CSVs or mix old runs.

In [ ]:
# Serializable layers replace shape-ambiguous tf.where/remasking Lambdas.
# This remains gated MIL pooling, not Q/K/V self-attention.
def build_window_encoder(cfg,n_ch=18):
    return nb.build_window_encoder(cfg,WIN,n_ch)

def build_window_classifier(cfg):
    return nb.build_window_classifier(cfg,WIN)

def build_mil_model(window_model,cfg):
    # ValidWindowEncoder gathers real windows BEFORE BatchNorm.
    # MaskedSoftmax has an explicit [batch, windows] output contract,
    # excludes invalid windows, renormalizes and rejects all-padded recordings.
    return nb.build_mil_model(window_model,cfg,WIN)

SPACE_RESMIL=dict(channels=[32,48],kernels=[(3,7,15),(5,9,17)],se_ratio=[4,8],
    embedding_dim=[32,64],spat_drop=[.1,.2],drop=[.25,.4],l2=[1e-4,1e-3],
    fine_tune=['frozen','last'],lr_mil=[1e-4,3e-4])

def parameter_report(cfg):
    return nb.model_parameter_report(FAMILY,cfg,WIN,LP_ORDER)

MODEL_READY=False
smoke_cfg={k:v[0] for k,v in SPACE_RESMIL.items()}
for mode in ('frozen','last'):
    print(nb.smoke_check(FAMILY,{**smoke_cfg,'fine_tune':mode},WIN))
MODEL_READY=True

In [ ]:
# Training details are in the pinned, tested nb.NotebookExperiment implementation.
# Stage 1: participant-balanced window loss.
# Stage 2: participant-balanced recording bags, padded windows excluded before encoding.
# Core controls alone fit LP during early stopping/candidate assessment.
# A final fixed-epoch refit uses all outer-training participants only.
# Both stages have atomic per-epoch model/optimizer/history/counter recovery.
print('Bounded-memory window and MIL bag batches; no unsafe Lambdas or obsolete Keras fit keywords.')

In [ ]:
# Unique configurations and inner participant folds are fixed independently of training seed.
# Each inner training split has an additional participant-disjoint early-stopping core.
# Inner assessment folds select configuration; only outer-training data select final refit epochs.
# Completed inner assessments are checkpointed separately from outer tasks.
preview=nb.unique_candidates(SPACE_RESMIL,N_TRIALS,SEARCH_SEED)
show_table(pd.DataFrame(preview),'Outer-fold-0 candidate preview (not model results)')
print('No reference model is retrained; no outer participant selects configuration or epochs.')

In [ ]:
if RUN_EXPERIMENT:
    assert MODEL_READY,'Run the model/mask/checkpoint smoke checks first'
    if RESTORE_FROM_ZIP:
        print('Restored:',nb.restore_checkpoint_bundle(RESTORE_FROM_ZIP,CHECKPOINT_DIR))
        RESTORE_FROM_ZIP=''
    experiment=nb.NotebookExperiment(globals(),FAMILY)
    run_result=experiment.nested(max_tasks=MAX_NEW_TASKS)
    globals().update(nb.unpack_result(globals(),run_result,FAMILY))
    print('Completed outer tasks:',len(run_result['selections']),'/',len(TRAIN_SEEDS)*N_FOLDS)
    print('Experiment complete:',EXPERIMENT_COMPLETE)
else:
    EXPERIMENT_COMPLETE=False
    print('Set RUN_EXPERIMENT=True to start/resume. Epoch and task checkpoints are automatic once training starts.')

In [ ]:
if EXPERIMENT_COMPLETE:
    globals().update(nb.summary_tables(globals(),FAMILY))
    show_table(summary.round(4),'Complete pooled recording and participant OOF metrics by seed')
    show_table(participant_ci.round(4),'Participant-stratified bootstrap intervals; conditional on fitted predictions')
    show_table(participant_fold_metrics.round(4),'Participant fold metrics; fold SD is descriptive')
    show_table(selection_table,'Selected configurations, determined inside outer training',limit=100)
    show_table(trial_table,'All inner candidate assessments',limit=100)
    plot_oof_diagnostics(nested_oof,['family','seed'])
else:
    print('Complete all outer tasks before inspecting test performance.')

In [ ]:
paired_comparisons=pd.DataFrame()
if EXPERIMENT_COMPLETE:
    globals().update(nb.error_reports(globals(),FAMILY))
    if REFERENCE_OOF_PATH.is_file():
        # Fail visibly on mismatched cohort, labels, folds or duplicate reference rows.
        paired_comparisons=nb.paired_tables(globals(),FAMILY,REFERENCE_OOF_PATH)
        show_table(paired_comparisons.round(4),'Paired LP-ResMIL minus completed reference (shared seeds only)')
    else:
        print('No reference OOF file supplied; no model is retrained and no comparison is invented.')
else:
    print('Reference comparisons wait for all outer tasks to finish.')

In [ ]:
if EXPERIMENT_COMPLETE:
    globals().update(nb.error_reports(globals(),FAMILY))
    show_table(participants.sort_values('nll',ascending=False).round(4),'Complete participant OOF table',limit=200)
    show_table(participant_errors.round(4),'Misclassified participants',limit=200)
    show_table(pd_severity.round(4),'PD sensitivity by verified HY stage')
    show_table(by_study.round(4),'Study-by-diagnosis metrics')
    show_table(seed_stability.round(4),'Persistent errors and probability variability across seeds')
    show_table(inner_loss.head(50).round(4),'Selected inner-assessment NLL contributors; repeated people across outer folds are not independent')
    show_table(attention_audit.round(6),'Valid-window attention sums (must equal one)',limit=100)
    if not attention_oof.empty:
        top=attention_oof.sort_values('attention',ascending=False).groupby(['family','seed','fold','file'],group_keys=False).head(3)
        show_table(top.round(5),'Three highest-weight valid windows per test recording',limit=100)
    print('Attention weights describe model weighting, not causal sensor importance or percentage of discriminative power.')
else:
    print('Diagnostic tables wait for the complete experiment.')

In [ ]:
EXPORT_CHECKPOINT_BUNDLE=False
EXPORT_ANALYSIS_CSV=False
if EXPORT_CHECKPOINT_BUNDLE:
    print('Checkpoint ZIP:',nb.export_checkpoint_bundle(CHECKPOINT_DIR))
    print('Download this ZIP or retain it on persistent storage before ending the runtime.')
if EXPORT_ANALYSIS_CSV:
    if not EXPERIMENT_COMPLETE: raise RuntimeError('Analysis export requires complete OOF results')
    nb.export_tables(globals(),FAMILY)
if not EXPORT_CHECKPOINT_BUNDLE and not EXPORT_ANALYSIS_CSV:
    print('No extra export requested. Automatic epoch/task checkpoints remain in',CHECKPOINT_DIR)

## Reporting checklist and limits

Report every declared seed, all outer folds, the inner-search budget and selected epoch counts. The corrected MIL encoder excludes padding before BatchNorm; this changes training relative to the old TimeDistributed implementation and requires new checkpoints. The supported regularizer API is explicit, not a claim of exact legacy TensorFlow numerical reproduction.

Per-participant predictions average recording probabilities. HY/study subgroup counts and missingness must accompany metrics. Attention exports are model weights, not causal evidence or percentages of discriminative power. Repeated inner-validation participants across outer folds are not independent samples.

Optional comparisons use existing OOF files only and require identical participant, diagnosis and fold assignments. No reference model is retrained. Conditional bootstrap intervals do not include all training/split variability or external-generalization uncertainty.

For safety, epoch checkpoints include full model/optimizer state, history and stopping/LR counters; task records commit atomically and are tied to the dataset/source/configuration. Full runtime resets still require persistent storage or an explicitly exported checkpoint ZIP. Old CSV-only checkpoints are not migrated or silently accepted.

No full Kaggle/Colab cohort training is executed as part of this compatibility update. Run the actual model/save-load smoke checks on the selected cloud runtime before enabling training.